In [1]:
#Importing the standard libraries
import numpy as np
np.set_printoptions(legacy='1.25',suppress=True)

import matplotlib.pyplot as plt
%matplotlib qt

import time
import warnings
warnings.filterwarnings('ignore')
import datetime



In [2]:
#Importing the solver modules
from BaF_solver.system import System



ModuleNotFoundError: No module named 'BaF_solver.interaction'

In [3]:
from src.states import SigmaLevel,PiLevelParity

In [4]:
from src.obe import obe 
from src.obe import Excitation 

In [5]:
from src.obe_with_gradient import Excitation
from src.obe_with_gradient import obe

In [6]:
#Reference ordering

Bz = 20
b=system.System([0,2],['1/2-','3/2-','5/2-'],B_field = [0.0,0.0,Bz],ignore_mF = False)

start = time.perf_counter()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
b.pi_Hamiltonian.generate_bare()
b.pi_Hamiltonian.Zeeman.generate_Zeeman()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()
b.pi_Hamiltonian.diagonalize()


G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,3)
E_global =b.pi_Hamiltonian.diagonalized_states
EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,3)

print(f"Bz = {Bz}.")
print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

G = G_global#[0:96]
GH = GH_global#[0:96,0:96]
E = E_global[0:16]
EH = EH_global[0:16,0:16]

vec_prev_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
vec_prev_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

G0_mean = np.mean(np.diag(GH)[0:16])
G2_mean = np.mean(np.diag(GH)[16:80+16])
E_mean =  np.mean(np.diag(EH))

Bz = 20.
Static Hamiltonian took : 0.9560238340018259s. 


np.set_printoptions(legacy='1.25')
#Check the order of mF levels
max_sigma_state_array_ref = []
for i,sigma_states in enumerate(b.sigma_states):
    # I need to find, whcih of the diagonalized eigen states have the the maximum amplitude for the gien sigma state
    
    vec_prev = b.sigma_Hamiltonian.diagonalized_states_as_vectors #is a matricx wherre coliumns are eigne vectors
    #Check the ith entry of every vectors to see which one is the maximum
    test_vec = vec[:,i] 
    #find the index of the max
    max_idx = np.argmax(np.abs(test_vec))
    max_sigma_state_array_ref.append(max_idx)

print(max_sigma_state_array_ref)

max_pi_state_array_ref = []
for st in E:
    list_st = st.states
    list_amp = st.amplitude
    max_state_idx = np.argmax([np.abs(val) for val in list_amp])
    max_state = list_st[max_state_idx]
    max_pi_state_array_ref.append(max_state)
print(Bz, max_pi_state_array_ref)

In [7]:
GH_list = [] 
EH_list = [] 
H0_list = []
BR_list = []
Hint_1_list,Hint_2_list,Hint_3_list = [],[],[]

Bz_list = np.arange(20,30,1)#np.arange(1,31,0.25)
for Bz in Bz_list:
    b=system.System([0,2],['1/2-','3/2-','5/2-'],B_field = [0.0,0.0,Bz],ignore_mF = False)

    start = time.perf_counter()

    b.sigma_Hamiltonian.generate_bare()
    b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
    b.pi_Hamiltonian.generate_bare()
    b.pi_Hamiltonian.Zeeman.generate_Zeeman()

    #Next diagonalize the Hamiltonian for this system
    b.sigma_Hamiltonian.diagonalize()
    b.pi_Hamiltonian.diagonalize()


    G_global = b.sigma_Hamiltonian.diagonalized_states
    GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,3)
    E_global =b.pi_Hamiltonian.diagonalized_states
    EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,3)

    print(f"Bz = {Bz}.")
    print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

    G_temp = G_global
    GH_temp = GH_global
    E_temp = E_global[0:16]
    EH_temp = EH_global[0:16,0:16]

    vec_current_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
    vec_current_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

    #check the new ground state's overlap with the the previous ground states    permutation_sigma = []
    vec_current_hermit = vec_current_sigma.conj().T
    permutation_sigma = []
    for i in range(len(G_temp)):
        #check current ith state ground state
        temp = vec_current_hermit@vec_prev_sigma[:,i]
        max_idx = np.argmax(np.abs(temp))
        permutation_sigma.append(max_idx)

    #check the new excited state's overlap with the the previous excited states
    permutation_pi = []
    vec_current_hermit = vec_current_pi.conj().T
    for j in range(len(E_temp)):
        #check current jth state excited state
        temp = vec_current_hermit@vec_prev_pi[:,j]
        max_idx = np.argmax(np.abs(temp))
        permutation_pi.append(max_idx)


    G_new = [G_temp[i] for i in permutation_sigma]
    E_new = [E_temp[j] for j in permutation_pi]
    GH_new_diag = np.diag(GH_temp)[permutation_sigma]
    EH_new_diag = np.diag(EH_temp)[permutation_pi]
    GH_list.append(GH_new_diag)
    EH_list.append(EH_new_diag)

    H0 = np.diag(np.append(GH_new_diag,EH_new_diag))
    assert np.allclose(np.imag(H0),np.zeros(H0.shape))
    H0_list.append(H0.astype(np.float64))

    b.generate_branching_ratios(G_new,E_new)
    BR = 0.96*b.branching_ratios
    assert np.allclose(np.imag(BR), np.zeros(BR.shape))

    BR_list.append(BR.astype(np.float64))

    start  = time.perf_counter()
    
    b.generate_interaction_Hamiltonian(G_new,E_new)
    Hint_1=np.round(b.interaction_Hamiltonian,3)

    b.generate_interaction_Hamiltonian(G_new,E_new,pol=+1)
    Hint_2 = np.round(b.interaction_Hamiltonian,3)

    b.generate_interaction_Hamiltonian(G_new,E_new,pol=-1)
    Hint_3 = np.round(b.interaction_Hamiltonian,3)
    print(f'All Hamiltonians took : {time.perf_counter() - start} s.')

    Hint_1_list.append(Hint_1)
    Hint_2_list.append(Hint_2)
    Hint_3_list.append(Hint_3)

    #rearragne the roder of the the vectors recorded as matrices too
    vec_prev_sigma,vec_prev_pi = vec_current_sigma[:,permutation_sigma],vec_current_pi[:,permutation_pi]
    

Bz = 20.
Static Hamiltonian took : 0.024939541999629s. 
Pi branching took : 1.734696083000017 sec
Sigma+ branching took : 0.42126979200111236 sec
Sigma- branching took : 0.43507866700019804 sec
All Hamiltonians took : 1.3690916249979637 s.
Bz = 21.
Static Hamiltonian took : 0.02523920800013002s. 
Pi branching took : 0.42954891699992004 sec
Sigma+ branching took : 0.420940291001898 sec
Sigma- branching took : 0.432780916999036 sec
All Hamiltonians took : 1.2874294169996574 s.
Bz = 22.
Static Hamiltonian took : 0.0241994580028404s. 
Pi branching took : 0.42413545900126337 sec
Sigma+ branching took : 0.43142204099785886 sec
Sigma- branching took : 0.4150656670026365 sec
All Hamiltonians took : 1.3213450419971196 s.
Bz = 23.
Static Hamiltonian took : 0.024101667000650195s. 
Pi branching took : 0.4433232500014128 sec
Sigma+ branching took : 0.43466045899913297 sec
Sigma- branching took : 0.4448024160010391 sec
All Hamiltonians took : 1.269591500000388 s.
Bz = 24.
Static Hamiltonian took : 0

In [8]:
import scipy
from scipy.interpolate import interp1d,CubicSpline

# Given data
B_vals = np.arange(20,30,1)  # shape (10,)
H0_vals = np.array(H0_list)  # shape (10, 112, 112)

BR_vals = np.array(BR_list)

Hint_1_vals = np.array(Hint_1_list)
Hint1_max = np.amax(np.abs(Hint_1_vals),axis=0)

Hint_2_vals = np.array(Hint_2_list)
Hint2_max = np.amax(np.abs(Hint_2_vals),axis=0)

Hint_3_vals = np.array(Hint_3_list)
Hint3_max = np.amax(np.abs(Hint_3_vals),axis=0)

interpol_kind = 'cubic'

def interpolate(x,y,real_imag = False):
    if real_imag == True:
        
        y_interp_real = interp1d(x, np.real(y),     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        y_interp_imag = interp1d(x, np.imag(y),     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        #y_interp_real = CubicSpline(x, np.real(y))#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        #y_interp_imag = CubicSpline(x, np.imag(y))#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        y_interp = (y_interp_real,y_interp_imag)
    else:
        #y_interp = CubicSpline(x, y)#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        y_interp = interp1d(x, y, kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
    return y_interp


# Flatten for fast interpolation (shape: (10, 112*112))
H0_flat = H0_vals.reshape(len(B_vals), -1)
BR_flat = BR_vals.reshape(len(B_vals), -1)
Hint_1_flat = Hint_1_vals.reshape(len(B_vals), -1)
Hint_2_flat = Hint_2_vals.reshape(len(B_vals), -1)
Hint_3_flat = Hint_3_vals.reshape(len(B_vals), -1)


H0     = interpolate(B_vals, H0_flat)
BR     = interpolate(B_vals, BR_flat)
Hint_1 = interpolate(B_vals, Hint_1_flat,real_imag = True)
Hint_2 = interpolate(B_vals, Hint_2_flat,real_imag = True)
Hint_3 = interpolate(B_vals, Hint_3_flat,real_imag = True)



"""
def get_interp_array(A_interp,shape,t,real_imag = False):
    if real_imag:
        real, imag = A_interp
        return (real(t).reshape(shape).astype(np.complex128),imag(t).reshape(shape).astype(np.complex128))
    else:
        return A_interp(t).reshape(shape).astype(np.complex128)
"""

'\ndef get_interp_array(A_interp,shape,t,real_imag = False):\n    if real_imag:\n        real, imag = A_interp\n        return (real(t).reshape(shape).astype(np.complex128),imag(t).reshape(shape).astype(np.complex128))\n    else:\n        return A_interp(t).reshape(shape).astype(np.complex128)\n'

In [9]:
plt.imshow(Hint1_max.real)
plt.figure()
plt.imshow(np.abs(Hint_1_list[0]))

In [10]:
test = [Hint_1,Hint_2]
for item in test:
    print(item)

(<scipy.interpolate._interpolate.interp1d object at 0x150d38b40>, <scipy.interpolate._interpolate.interp1d object at 0x15093e620>)
(<scipy.interpolate._interpolate.interp1d object at 0x150608fa0>, <scipy.interpolate._interpolate.interp1d object at 0x151834c80>)



z = [br[11,2] for br in BR_list]

B_range= np.arange(20,30,0.1)
y = np.empty_like(B_range)

for i,B in enumerate(B_range):
    br = get_interp_array(BR,(len(G),len(E)),B,real_imag = False)
    y[i] = br[11,2] 

plt.plot(B_vals,z,'or')    
plt.plot(B_range,y)

In [11]:
np.sqrt(12544)

112.0

def get_interp_array(A_interp,shape,t,real_imag = False):
    if real_imag:
        real, imag = A_interp
        return (real(t).reshape(shape).astype(np.complex128),imag(t).reshape(shape).astype(np.complex128))
    else:
        return A_interp(t).reshape(shape).astype(np.complex128)

n_total = len(G) + len(E)
H0_base = get_interp_array(H0,(n_total,n_total),20,real_imag = False)
H = get_interp_array(H0,(n_total,n_total),21.99,real_imag = False) - \
    H0_base
print(f"h max : {np.amax(H)}, H min : {np.amin(H)}")

temp = [item for item in enumerate(np.arange(1,31,0.25))]
print(temp)

for (i,B) in temp: 
    err = np.abs(H0_list[i] - get_interp_array(H0_interp,np.shape(H0_list[0]),B))
    print(f"B : {B}, error = {np.sum(err) :.3f}, max error : {np.amax(err) :.3f}")

import matplotlib.colors as colors
B = np.arange(20,30,0.1)
en = []
for i,Bval in enumerate(B):
    #en.append(H0_list[i])
    #plt.imshow(B,en,'.-');
    A = get_interp_array(Hint_2_interp,Hint_2.shape,Bval,real_imag = True)[0]
    norm = colors.LogNorm(vmin=1e-3, vmax=1)#np.abs(A.max() - A.min()))
    plt.clf()
    plt.imshow(A)#,norm=norm)
    plt.pause(0.1)

In [13]:


#to select a desired set of transitions
Gamma = 2*np.pi*2.7
tsigma = 8.192/4
import random
def generate_transition(desired_transitions,polarization):
    
    field = []
    for item in desired_transitions:
        (i,j) = item
        rabi = random.uniform(0.1, 5.0)*Gamma
        pol = polarization
        groundState = G[i]
        excitedState = E[j]
        det = random.uniform(-1,1)*Gamma 
        pos = 0
        dia = 4*tsigma
        temp_field = Excitation(rabi,pol,groundState,excitedState,detuning = det,position = pos,diameter = dia ,shape = "Uniform")
        field.append(temp_field)
    
    return field,desired_transitions

In [14]:
def flatten(x):
    temp_list = []
    for item in x:
        if type(item) == list:
            for temp_item in item:
                temp_list.append(temp_item)
        else:
            temp_list.append(item)
    return temp_list


In [15]:
### only when we want to force a choice of transition

#desired_Z = [(0, 14), (1, 1), (2, 5), (3, 11), (4, 13), (5, 5), (6, 3), (7, 2), (8, 1), (9, 10), (10, 0), (11, 3), (12, 5), (13, 4), (15, 15), (17, 15), (18, 9), (19, 13), (20, 12), (21, 15), (22, 11), (23, 14), (24, 10), (25, 13), (27, 2), (28, 11), (29, 10), (30, 0), (31, 15), (32, 1), (33, 2), (34, 0), (35, 3), (36, 1), (37, 10), (38, 2), (39, 3), (40, 7), (41, 1), (42, 3), (43, 2), (44, 1), (45, 14), (46, 1), (47, 5), (48, 7), (49, 11), (50, 0), (51, 1), (52, 2), (53, 1), (54, 3), (55, 10), (56, 5), (57, 3), (58, 5), (59, 1), (60, 1), (61, 0), (62, 0), (63, 15), (64, 10), (65, 6), (66, 7), (68, 8), (69, 10), (70, 11), (71, 9), (72, 12), (73, 15), (74, 13), (75, 14), (76, 15), (79, 10), (80, 6), (81, 7), (83, 13), (85, 14), (86, 10), (87, 15), (88, 11), (90, 12), (91, 13), (92, 14), (93, 15)]
#desired_X = [(0, 13), (1, 14), (2, 1), (3, 5), (4, 5), (5, 1), (6, 7), (7, 13), (8, 14), (9, 3), (10, 4), (11, 5), (12, 4), (13, 5), (15, 9), (16, 15), (17, 9), (18, 13), (19, 14), (20, 13), (21, 14), (22, 12), (23, 15), (24, 11), (25, 14), (26, 10), (27, 6), (28, 7), (29, 11), (30, 1), (31, 0), (32, 0), (33, 1), (34, 1), (35, 2), (36, 2), (37, 3), (38, 4), (39, 5), (40, 1), (41, 0), (42, 2), (43, 1), (44, 0), (45, 1), (46, 0), (47, 4), (48, 1), (49, 2), (50, 1), (51, 0), (52, 1), (53, 5), (54, 5), (55, 6), (56, 1), (57, 2), (58, 3), (59, 0), (60, 0), (61, 1), (62, 1), (63, 0), (64, 3), (65, 5), (66, 6), (67, 10), (68, 9), (69, 6), (70, 10), (71, 13), (72, 13), (73, 14), (74, 14), (75, 13), (76, 14), (77, 15), (78, 10), (79, 6), (80, 10), (81, 13), (83, 14), (84, 10), (85, 13), (86, 11), (87, 14), (88, 10), (89, 15), (90, 13), (91, 14), (92, 13), (93, 14), (94, 15)]
level_to_optimize = 15
desired_Z = [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 2), (8, 1), (9, 0), (10, 10), (11, 3), (12, 2), (13, 1), (14, 0), (17, 15), (18, 14), (19, 13), (20, 12), (21, 11), (22, 10), (24, 15), (25, 14), (26, 13), (27, 2), (28, 3), (29, 10), (30, 15), (31, 0), (32, 1), (33, 2), (34, 0), (35, 3), (36, 10), (37, 1), (38, 2), (39, 3), (40, 7), (41, 1), (42, 3), (43, 2), (44, 1), (45, 0), (46, 1), (47, 7), (48, 3), (49, 2), (50, 1), (51, 0), (52, 2), (53, 4), (54, 6), (55, 2), (56, 1), (57, 10), (58, 0), (59, 3), (60, 2), (61, 1), (62, 0), (63, 15), (64, 10), (65, 6), (66, 12), (67, 13), (68, 14), (69, 15), (71, 10), (72, 11), (73, 12), (74, 13), (75, 14), (76, 15), (79, 10), (80, 11), (81, 12), (82, 13), (83, 14), (84, 15), (88, 10), (89, 11), (90, 12), (91, 13), (92, 14), (93, 15)]
desired_X = [(0, 13), (1, 14), (2, 13), (3, 12), (4, 0), (5, 3), (6, 7), (7, 1), (8, 0), (9, 15), (10, 11), (11, 12), (12, 4), (13, 5), (14, 4), (16, 15), (17, 14), (18, 15), (19, 14), (20, 13), (21, 12), (22, 11), (23, 10), (24, 14), (25, 15), (26, 14), (27, 3), (28, 12), (29, 11), (30, 0), (31, 1), (32, 0), (33, 3), (34, 15), (35, 2), (36, 3), (37, 2), (38, 3), (39, 2), (40, 1), (41, 0), (42, 5), (43, 1), (44, 0), (45, 1), (46, 0), (47, 3), (48, 2), (49, 1), (50, 0), (51, 1), (52, 3), (53, 0), (54, 7), (55, 1), (56, 0), (57, 3), (58, 1), (59, 2), (60, 1), (61, 0), (62, 1), (63, 0), (64, 6), (65, 10), (66, 11), (67, 12), (68, 15), (69, 0), (70, 10), (71, 11), (72, 10), (73, 11), (74, 12), (75, 13), (76, 14), (77, 15), (78, 10), (79, 6), (80, 7), (81, 11), (82, 12), (83, 13), (84, 14), (85, 15), (87, 10), (88, 11), (89, 10), (90, 11), (91, 12), (92, 13), (93, 14), (94, 15)]

current_field,transitions_Z = generate_transition(desired_Z,0)
N_z= len(current_field)
temp_field,transitions_X = generate_transition(desired_X,1) 
current_field.append(temp_field)
N_sigma = len(current_field[-1])
current_field=flatten(current_field)


In [16]:
#Count the number of transitions from the different N 
# We are going to address both G levels within the same N with the output from same laser.
Nz_0,Nz_2 = 0,0
for item in transitions_Z:
    g = item[0]                 #index of the ground state
    
    max_N_idx = np.argmax(np.abs(G[g].amplitude))
    N_g = G[g].states[max_N_idx].N

    if N_g == 0:
        Nz_0 += 1
    elif N_g == 2:
        Nz_2 += 1
    else:
        warnings.warn('Warning Message: State with N = 0 or N = 2 expected.') 
print(Nz_0,Nz_2)


Nx_0,Nx_2 = 0,0
for item in transitions_X:
    g = item[0]                 #index of the ground state
    
    max_N_idx = np.argmax(np.abs(G[g].amplitude))
    N_g = G[g].states[max_N_idx].N

    if N_g == 0:
        Nx_0 += 1
    elif N_g == 2:
        Nx_2 += 1
    else:
        warnings.warn('Warning Message: State with N = 0 or N = 2 expected.') 
print(Nx_0,Nx_2)


Ns = [Nz_0,Nz_2,
        Nx_0,Nx_2]


15 70
15 78


In [17]:
# Temporary function to get the position of the beaM GIVEN THE SEQUENCE AND DIAMETER OF THE BEAM
def get_pos(A):
    cumsum_arr = np.cumsum(A)
    pos = [A[0]/2]
    for i in range(1,len(A)):
        temp_pos = np.round((cumsum_arr[i]+cumsum_arr[i-1])*0.5,2)
        pos.append(temp_pos)
    return pos

In [18]:
import traceback
Bz = 20
#The cost function. Returns the population of the desired state.
import copy
def r22(rabi_det_params:list,Ns:list,Omegas:tuple,passes=1):
    try:
        Omega_z_0,Omega_z_2, z_x_factor = Omegas
        Omega_x_0,Omega_x_2 = z_x_factor*Omega_z_0,z_x_factor*Omega_z_2
        package = 'Python'
        obe_mode = 'symengine'
        [Nz_0,Nz_2,Nx_0,Nx_2] = Ns
        
        P_z_0 = []
        P_z_2 = []
        P_x_0 = []
        P_x_2 = []

        for i in range(len(current_field)): #Converting between Nevergrad array and field rabis and detunings
                current_field[i].rabi = rabi_det_params[i][0]
                current_field[i].detuning = rabi_det_params[i][1]
                if i < Nz_0:
                    P_z_0.append(rabi_det_params[i][0]**2)
                elif i >= Nz_0 and i < (Nz_0 + Nz_2):
                    P_z_2.append(rabi_det_params[i][0]**2)
                elif i >= (Nz_0 + Nz_2) and i < (Nz_0 + Nz_2 + Nx_0):
                    P_x_0.append(rabi_det_params[i][0]**2)
                else:
                    P_x_2.append(rabi_det_params[i][0]**2)
        
        total_pow_z_0 = np.sum(P_z_0)
        total_pow_z_2 = np.sum(P_z_2)
        total_pow_x_0 = np.sum(P_x_0)
        total_pow_x_2 = np.sum(P_x_2)
        
        
        P_z_0 = np.array(P_z_0)/np.sum(P_z_0)
        P_z_2 = np.array(P_z_2)/np.sum(P_z_2)
        P_x_0 = np.array(P_x_0)/np.sum(P_x_0)
        P_x_2 = np.array(P_x_2)/np.sum(P_x_2)
        
        
        threshold = 0.01 #rise time of the pulse signal used
        P_z_0[P_z_0 < threshold] = 0
        P_z_2[P_z_2 < threshold] = 0
        P_x_0[P_x_0 < threshold] = 0
        P_x_2[P_x_2 < threshold] = 0

        total_pow_z_0 = np.sum(P_z_0)
        total_pow_z_2 = np.sum(P_z_2)
        total_pow_x_0 = np.sum(P_x_0)
        total_pow_x_2 = np.sum(P_x_2)        
        
        P_z_0 = np.array(P_z_0)/total_pow_z_0*4*tsigma
        P_z_2 = np.array(P_z_2)/total_pow_z_2*4*tsigma
        P_x_0 = np.array(P_x_0)/total_pow_x_0*4*tsigma
        P_x_2 = np.array(P_x_2)/total_pow_x_2*4*tsigma        

        pos_z_0 = np.round(get_pos(P_z_0),5)
        pos_z_2 = np.round(get_pos(P_z_2),5)
        pos_x_0 = np.round(get_pos(P_x_0),5)
        pos_x_2 = np.round(get_pos(P_x_2),5)


        field = current_field
        field_Z_0 = field[0:
                            Nz_0]
        field_Z_2 = field[Nz_0:
                                Nz_0+Nz_2]
        field_X_0 = field[Nz_0+Nz_2:
                                    Nz_0+Nz_2+Nx_0]
        field_X_2 = field[Nz_0+Nz_2+Nx_0:
                                        Nz_0+Nz_2+Nx_0+Nx_2]
    
        #Modify fields for N = 0 
        ############################
        for i in range(len(field_Z_0)):
            field_Z_0[i].position = pos_z_0[i]
            field_Z_0[i].diameter = P_z_0[i]
            field_Z_0[i].rabi = Omega_z_0

        Z0_field = []
        for item in field_Z_0:
            Z0_field.append(item)
            item_other_sideband = copy.copy(item)
            #determine frequency to shift by
            #What is the frequency of this light 
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G0_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            Z0_field.append(item_other_sideband)

        for i in range(len(field_X_0)):
            field_X_0[i].position = pos_x_0[i]
            field_X_0[i].diameter = P_x_0[i]
            field_X_0[i].rabi = 1/np.sqrt(2)*Omega_x_0

        X0_field = []
        for item in field_X_0:
            X0_field.append(item)
            
            #pol reversed item
            item_pol_reversed = copy.copy(item)
            item_pol_reversed.pol *= -1
            X0_field.append(item_pol_reversed)

            #other sideband
            item_other_sideband = copy.copy(item)
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G0_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            X0_field.append(item_other_sideband)

            #other sideband pol reversed
            item_other_sideband_pol_reversed = copy.copy(item_other_sideband)
            item_other_sideband_pol_reversed.pol *= -1
            X0_field.append(item_other_sideband_pol_reversed)



        #Modify fields for N = 2
        ######################### 
        for i in range(len(field_Z_2)):
            field_Z_2[i].position = pos_z_2[i]
            field_Z_2[i].diameter = P_z_2[i]
            field_Z_2[i].rabi = Omega_z_2

        Z2_field = []
        for item in field_Z_2:
            Z2_field.append(item)
            item_other_sideband = copy.copy(item)
            #determine frequency to shift by
            #What is the frequency of this light 
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G2_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            Z2_field.append(item_other_sideband)

        for i in range(len(field_X_2)):
            field_X_2[i].position = pos_x_2[i]
            field_X_2[i].diameter = P_x_2[i]
            field_X_2[i].rabi = 1/np.sqrt(2)*Omega_x_2

        X2_field = []
        for item in field_X_2:
            X2_field.append(item)
            
            #pol reversed item
            item_pol_reversed = copy.copy(item)
            item_pol_reversed.pol *= -1
            X2_field.append(item_pol_reversed)

            #other sideband
            item_other_sideband = copy.copy(item)
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G2_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            X2_field.append(item_other_sideband)

            #other sideband pol reversed
            item_other_sideband_pol_reversed = copy.copy(item_other_sideband)
            item_other_sideband_pol_reversed.pol *= -1
            X2_field.append(item_other_sideband_pol_reversed)
        

        steps=100
        n=len(E)+len(G)
        r_init = np.zeros((n,n),dtype = np.complex128)
        #r_init[15,15] = 1.0+0.0j
        
        for i in range(len(G)): #Initializing the density matrix considering a rot temperature of 4 K
            if i<16:
                r_init[i,i] = 1.0#/len(G)
            else:
                r_init[i,i] = 0.628*1.0#/len(G)

        
        pops = []
        Hint_Z = None
        Hint_X = None
        test_factor = 30
        #print('Creating obes')

        start = time.perf_counter()
        my_obe_1 = obe(Z0_field + Z2_field,[G,E],H0,Hint_1,BR,[Bz,0.0065*5],test_factor,mode = obe_mode,max_Hints = [Hint1_max])
        
        print(f"Z Hamiltonian took {time.perf_counter()-start}s")
        start = time.perf_counter()
        my_obe_2 = obe(X0_field + X2_field,[G,E],H0,[Hint_2,Hint_3],BR,[Bz,0.0065*5],test_factor,mode = obe_mode,max_Hints = [Hint2_max,Hint3_max])
        print(f"X Hamiltonian took {time.perf_counter()-start}s")

        method = 'RK45'
        for count in range(passes):
            start = time.perf_counter()
            ans = my_obe_1.solve(steps,r_init,
                                max_step_size = 1/Gamma,#max_step_size,
                                package = package,
                                method = method)
            print(f"Z Solve took {time.perf_counter() - start :.3f}.")
            pops.append(ans)
            rho = np.array(ans[-1]) #gives the solution at the end of the time
            r_init = rho.reshape(n,n)
            #print(f"Z solve took {time.time()-start} s.")
            
            
            r_diag = np.round(np.diag(r_init),3)
            total_pop = np.sum(r_diag)
            sorted_indices = np.argsort(r_diag)[::-1]
            print(f"Pass Z:{count}",end=",")
            for indx in range(6):
                print(f"r{sorted_indices[indx]} = {r_diag[sorted_indices[indx]].real}",end = ",")
            print(f"total_pop = {total_pop}")

            start = time.perf_counter()
            ans = my_obe_2.solve(steps,r_init,
                                max_step_size = 1/Gamma,
                                package=package,
                                method = method)
            print(f"X Solve took {time.perf_counter() - start :.3f}.")
            pops.append(ans)
            rho = np.array(ans[-1]) #gives the solution at the end of the time
            r_init = rho.reshape(n,n)
            
            r_diag = np.round(np.diag(r_init),3)
            total_pop = np.sum(r_diag)
            
            sorted_indices = np.argsort(r_diag)[::-1]
            print(f"Pass X:{count}",end=",")
            for indx in range(6):
                print(f"r{sorted_indices[indx]} = {r_diag[sorted_indices[indx]].real}",end = ",")
            print(f"total_pop = {total_pop.real}")
            
    
        r_level2optimize=np.real(r_init[level_to_optimize,level_to_optimize])
        rgg = 0

        for temp_count in range(len(G)):
            if temp_count != level_to_optimize:
                rgg += np.real(r_init[temp_count,temp_count])
        
        
        for i in range(len(G)):
            print(f"\'r_{i}\': {np.round(r_init[i,i],5)}",end = ",")
        print("----------")
        
        return -r_level2optimize,pops #nevergrad minimizes
    except Exception as e:
        print(e)
        traceback.print_exc()
        print(rabi_det_params,Omegas)
        print("Error encountered with this input. Default value of 1 returned.")
        return 1.0,1.0

In [19]:
#f= open('../Bz_variation/XZ_rabi_det_N0_14_Bz_20.0_tsigma_2.048_passes_9_evolution_office_CMA.csv')
f= open('../Bz_variation/XZ_rabi_det_N0_15_Bz_20.0_tsigma_2.048_passes_9_evolution_mac_CMA.csv') 
all_array = []
for i,line in enumerate(f):
    if i < 3:
        continue
    mylist = eval(line)
    all_array.append(mylist)
sorted_array = sorted(all_array, key=lambda item: item[0],reverse=True)
pop = [item[0] for item in all_array]
#plt.plot(pop,'-o')

In [20]:
#arr = [-0.06794000930252098, (0.09073297687597676, -2.273033490125983), (1.422914889894515, -3.12866683381516), (2.0201918004489525, -0.6466940623066644), (2.1820373913369875, -2.9922987504997343), (1.587582684083073, -4.779832681581274), (0.6052025580529146, -3.517216140162518), (1.2085875845512954, -0.17035334723423454), (0.9253837313668611, 4.23577247140647), (0.18855871348290562, -3.745067422313548), (0.7667336526367492, 2.4519138309457897), (0.44046729243779886, -4.007941631942406), (0.6923217385593049, 2.1064145374621304), (1.157971473291706, 4.964374963730084), (3.6145493886817386, 1.067203235721742), (0.03799692778684676, -2.9329426680272492), (1.5723658910864438, 7.024505688988581), (2.501509350604561, 0.011040121333689807), (1.466832038489734, 4.4711490542032495), (0.5055610791485117, -1.7000413037717126), (0.786387301998399, 6.008626290135908), (0.7783688348496669, -0.18681766384374063), (0.1644155589080084, -2.5621698513401947), (1.1005815035768431, -1.1010740093490978), (2.6966079851669895, 5.604624213313732), (0.96729106232432, 1.880691471433173), (1.2230703136648238, 1.061196579401179), (0.8433524051571928, 1.5358719243840049), (1.7129094395007918, -2.7241441042414407), (0.2555747079634495, -7.584169170567547), (0.7080320268989198, -3.814646461724501), (0.021049955228157773, 3.0071931121467976), (0.7289276737688446, 4.331022099965598), (1.5754192872417363, 5.939603761046079), (1.0805057532620725, 1.476163129304841), (1.6130102892222562, 1.6982890050560062), (1.716301269606432, -3.5794192035559504), (1.1968250088264374, -13.27066667996463), (0.6258671068255195, -6.449632068938049), (0.20310026270297954, -3.6187879126852307), (1.079521368752466, 4.797067465917514), (0.37959928248407737, 3.6600429784243365), (1.0156854528201147, -2.8044991202557874), (0.18428892242321293, 1.239280928415368), (0.8626171380016505, 5.577215867772588), (1.8671877176569764, 3.3007740557658543), (0.5874596658687915, 0.5473610719002729), (1.9383358199736023, -4.8097944991982535), (0.44313004058940086, -2.79853156630245), (0.5173237744041382, -1.083970138392949), (1.9045831368846948, -0.7823111511314355), (0.5287378968884566, 9.277569784291478), (0.15192698778658814, 5.270893339227397), (1.2127669725059222, -4.5465833924340044), (0.3916955109355289, 2.3230348380945007), (1.6202377988803642, 2.201290536184822), (0.29409767990700714, 1.4589854589951186), (1.100570657806805, -4.758702895653554), (1.7257902239770009, 3.5885204594583566), (3.6214254479238797, 2.9591230936263613), (1.0040354944342094, 2.1895427790651123), (0.8368513898627776, -2.9991221819433003), (1.6681547772365808, 7.2959906562130055), (1.651133246753962, -2.7111256108972794), (1.4500543372781218, 8.397035852663201), (1.5405620460236067, -1.4437773315076767), (1.995982054819311, 3.044770070497677), (1.889802196090565, 2.020872323247071), (0.8712155212268656, -0.616539134102678), (1.182849190751816, -6.715062731068209), (2.5939025213561777, -6.828849662101289), (2.165158933707325, -4.035880113335514), (1.1841036985068665, -1.930072274588714), (0.5801509360462682, 12.19744255052058), (1.749330678795475, 1.9506920987692742), (1.1949129980521123, 1.7462597142725136), (0.4125718936599452, -1.1769335248565), (1.299215981762497, 5.168358941064932), (1.837642784289406, -3.1864500936782356), (0.1424274191293489, 0.5628539858855002), (0.05654506282804623, 4.916248208326636), (0.7017183759794897, -9.119612483641522), (1.4571869752703261, -7.958885712251331), (0.9624182118921467, 1.9899391967300364), (2.082639066153985, 4.327997118585684), (1.26514115544736, -6.927256727120644), (0.5140016162614581, -6.4987516002927075), (0.1169482415255914, -3.427022015801195), (1.4434114992926974, 2.9811537463436886), (1.0836016193325961, 8.699448424038888), (0.960379260378204, -0.6176088872493447), (0.9088143188502389, 0.3772018897158157), (0.17803984004142398, -1.5117189910711386), (0.9277672155422806, 3.381023825944648), (0.10391071835926706, 5.270674255277814), (0.5293269552070505, -4.897732211284179), (3.658095047561858, -4.360929422070814), (1.8543177031677174, -0.9423959424919761), (2.3625448676790066, 1.3571609058281724), (0.8534164556680625, 2.2693528144716426), (0.5476905916017321, 2.090545204630735), (0.08002587311796305, 3.0085035922949137), (1.6887626277317493, 2.157590486865952), (0.3695484782896069, 0.48075280187015934), (0.3860374707688181, 3.327822525346557), (1.170135526213319, 2.918630793761639), (0.8644522311173631, 3.967095458194705), (1.3346404479465226, 2.0093590031688224), (0.8103473753645314, -4.836158160619684), (0.686776452554268, 0.7629274791618968), (1.6598068764387812, 6.5690704642271065), (3.504701664862317, 6.101081296646262), (1.0118884061445192, -6.124059619188825), (0.38549992558738944, -2.3333669421939076), (1.3911317172495006, 2.6022372687416193), (0.039651077536118295, 6.734460039035152), (0.9714317181892316, -10.346241034124898), (1.5291804004229068, -2.3069034245156264), (0.43239293831356784, -4.576325782342108), (0.12134484221004313, 10.860386873450873), (0.23523963457434158, -0.6836553383172294), (2.572685921993911, 7.387747542657661), (0.12945341583016867, 1.204139971256047), (3.447524901106365, -3.269693195488022), (0.5238400843938644, -2.4299973494887888), (1.6177651165049858, -2.9928395040378204), (1.128068286662358, -5.396516148845301), (0.9125779386588752, -2.056969730726187), (1.3102320306168624, -5.064008556024956), (1.005008877691171, 4.66537772326928), (0.3632339764452425, 2.141292985932944), (0.27731274419885976, -4.620006899480542), (2.021876698853699, -0.6380410351092731), (0.12428762445106821, 0.3196023125901843), (1.4036199930401128, -0.41079117983711216), (2.4050392324606347, -0.9330085938454986), (0.5353006539816636, 8.96885778456409), (1.9042069624565077, -6.834272883203064), (2.479407379465764, -3.197668698584803), (1.6668828978328332, -4.464977701415675), (1.9449955478707683, 1.7803203407210442), (2.4372241357823805, 2.5327574172240306), (1.3242356518564198, 4.993397753045883), (0.45734536120831193, 0.2937699901250704), (1.609471283602035, -3.662048556261028), (0.04521884545398842, 1.1957442866640522), (0.45433123944100345, -3.209225994061415), (1.5060702922441913, -6.796723657102415), (1.3811647340544533, 5.690825531282342), (2.7769093169885517, -3.9368682724935224), (0.5625021059124233, 3.1255153141935765), (0.2245461228891258, 0.3433261516471288), (0.6356911109740375, 2.0664157204736044), (0.7105826443300073, -4.268755577537629), (2.770206311040624, 3.663550206161267), (0.5144826856747674, 0.27209905181819166), (0.03893034731183181, -0.04767976184365003), (0.9591837991601687, -1.6841584826756024), (0.7929870683494654, 1.2843183273769496), (0.04631892239003826, 3.101559739327982), (0.6347117224095844, -2.8801811572514), (2.2422695319272314, -1.079960553981822), (3.149582383033587, -3.9741759388970332), (0.9220774165990389, -4.5009779878111065), (4.2044740237040505, 3.664748563324358, 3.0902152706874677, 4.824891064120179, 0.5821615732054538)]
arr = sorted_array[-1]
print(arr)
print(arr[-1])

[-35.21478098953151, (3.195300375504177, -8.752301830069891), (7.250432839097421, 13.868255695738478), (1.6270767101673878, 2.2714971134392568), (6.710557237097325, 1.5545285854562407), (8.815335852782216, 13.1785207815598), (3.4606234575083654, 9.799491589903626), (1.528230143333523, 0.4357321694759169), (4.646571264607843, -5.252687413269311), (2.007857469623084, 0.7661099570188927), (9.354746930127048, -6.76505647087024), (0.5088235141675782, 22.86498493837954), (9.116080761111608, -21.096700926330954), (8.413280010432993, -24.49349558386912), (4.9647109521824975, 2.0276216394794395), (1.1702684084738828, -8.648676844337524), (6.308101681516055, -7.373426095371076), (6.445224061568176, 2.9137332609051305), (6.932504399542282, -18.294003244987305), (5.016745433730354, 22.120759272178766), (3.690065755387027, 4.89501390377031), (5.567207094272462, -21.497974917654602), (6.953534671611963, 10.587216303898256), (4.4273263628065935, 4.020747096887417), (2.489245000494828, 9.5794039793489

In [21]:
#arr = sorted_array[-1]
rabi_det_array = arr[1:-1]
omegas = arr[-1]
print(omegas)
omegas_list = []
for i,item in enumerate(omegas):
    if i ==1:
        omegas_list.append(1*item)
    elif i == 2:
        omegas_list.append(1.*item)    
    else:
        omegas_list.append(1*item)
print(omegas_list)
#draw(rabi_det_array,Ns,tuple(omegas_list),passes=9)
val,pops = r22(rabi_det_array,Ns,tuple(omegas_list),passes=1)

(9.56637695183178, 28.43773786915947, 0.9357248781233761)
[9.56637695183178, 28.43773786915947, 0.9357248781233761]
Z Hamiltonian took 3.175725707998936s
X Hamiltonian took 7.854820999997173s
Solving started.
nfev: 7868
ODE solver took : 3.8100761669993517 s
Time spent on lambdified Hinit symengine= 0.642s
Time spent on interpolated Hinit scipy= 0.972s
Time spent on interpolated Hinit scipy symengine multiplication = 0.097s
Time spent on commutator numba = 0.699s
Time spent on decay = 0.100s
Time spent on repopulation = 0.114s
Z Solve took 3.920.
Pass Z:0,r15 = 4.334,r9 = 3.185,r8 = 2.37,r0 = 2.259,r12 = 2.252,r7 = 2.235,total_pop = (64.57300000000001+0j)
Solving started.
nfev: 7946
ODE solver took : 5.100635540999065 s
Time spent on lambdified Hinit symengine= 1.312s
Time spent on interpolated Hinit scipy= 1.646s
Time spent on interpolated Hinit scipy symengine multiplication = 0.197s
Time spent on commutator numba = 0.711s
Time spent on decay = 0.100s
Time spent on repopulation = 0.1

In [22]:
error

NameError: name 'error' is not defined

qt.qpa.backingstore: Back buffer dpr of 1 doesn't match <NSViewBackingLayer: 0x1455af810> contents scale of 2 - updating layer to match.
qt.qpa.backingstore: Back buffer dpr of 1 doesn't match <NSViewBackingLayer: 0x1455af810> contents scale of 2 - updating layer to match.


In [ ]:
import cProfile
import pstats

with cProfile.Profile() as pr:
    r22(rabi_det_array,Ns,omegas,passes=1)

stats = pstats.Stats(pr)
stats.sort_stats("tottime").print_stats(20)

## Plot the population evolution

In [ ]:
levels = 112
passes = 24
# Generate 112 colors from a continuous colormap
cmap = plt.get_cmap('hsv')  # Try 'nipy_spectral', 'turbo', or 'gist_rainbow' too
colors = [cmap(i / levels) for i in range(levels)]
import random
random.seed(110)
colors = [(random.random(), random.random(), random.random()) for _ in range(112)]
index = np.arange(levels)*(levels+1)
t_block = 8.192
x = np.arange(1000)*t_block/1000
for i in range(2*passes):
    M = pops[i]
    N = np.real(M[:,index])
    for j in range(levels):
        if i == 0:
            plt.plot(x,N[:,j],color = colors[j],label = str(j));
        elif i == 2*passes-1:
            plt.plot(x,N[:,j],color = colors[j]);
            plt.text(x[-1]+1,N[-1,j],str(j),fontsize = 7)
        else:
            plt.plot(x,N[:,j],color = colors[j]);
    x += t_block 
#plt.legend();


In [ ]:
ground_mF = [item.states[0].mF+np.linspace(-1/4,1/4,3) for item in G[0:16]]
ground_Energy = np.array([GH[i,i] for i in range(len(ground_mF))])
ground_Energy -= np.mean(ground_Energy)
min_ground_energy = np.amin(ground_Energy)
max_ground_energy = np.amax(ground_Energy)

excited_mF = [item.states[0].mF+np.linspace(-1/4,1/4,3) for item in E]
excited_Energy = np.array([EH[i,i] for i in range(len(E))])
excited_Energy -= np.mean(excited_Energy) - 1*np.abs(max_ground_energy-min_ground_energy)

"""
rabi_det_params = max_array # --------the solution from the optimization-----------------
for i in range(len(current_field)):
            current_field[i].rabi = rabi_det_params[i][0]
            current_field[i].detuning = rabi_det_params[i][1]
"""            
#Plot the energy levels
for i in range(len(ground_mF)):
    plt.plot(ground_mF[i],ground_Energy[i]+np.zeros(3))
    plt.text(ground_mF[i][1],ground_Energy[i]-8,f"G[{i}]",fontsize = 8)
for j in range(len(E)):
    plt.plot(excited_mF[j],excited_Energy[j]+np.zeros(3))
    plt.text(excited_mF[j][1],excited_Energy[j]-8,f"E[{j}]",fontsize = 8)
"""
rabis = []
for i in range(0,len(max_array),2):
    rabis.append(min_array[i])
max_rabi = np.amax(np.abs(np.array(rabis)))
    
# Plot the z transitions
for idx,item in enumerate(transitions_Z):
    i,j = item
    offset = random.uniform(-0.25,0.25)
    x = [ground_mF[i][1]+offset,excited_mF[j][1]+offset]
    y = [ground_Energy[i],excited_Energy[j]+current_field[idx].detuning]
    plt.plot(x,y,alpha = np.abs(current_field[idx].rabi/max_rabi),linewidth = np.abs(current_field[idx].rabi/max_rabi*5))


#Plot sigma transitions
for idx,item in enumerate(transitions_sigma):
    i,j = item
    offset = random.uniform(-0.05,0.05)
    x = [ground_mF[i][1]+offset,excited_mF[j][1]+offset]
    y = [ground_Energy[i],excited_Energy[j]+current_field[idx+N_z].detuning]
    #plt.plot(x,y,alpha = np.abs(current_field[idx+N_z].rabi/max_rabi),linewidth = np.abs(current_field[idx+N_z].rabi/max_rabi*10))
"""

In [ ]:
%load_ext cython

In [ ]:
%%cython -a

from states import SigmaLevel,PiLevelOmega

from spin_params import S,I1,I2
cdef double S_ = <double>S
cdef double I1_ = <double>I1
cdef double I2_ = <double>I2


from fast_wigners import wigner_6j,wigner_3j,wigner_9j

cdef extern from "math.h" nogil:
    double sqrt(double)
    double fabs(double)

cdef inline double kdel(double x,double y) nogil:
    return 1.0 if <int>(2*x) == <int>(2*y) else 0.0

cdef inline double reduced(double x) nogil:
    return sqrt(x*(x+1.0)*(2.0*x+1.0))

cdef inline double nreduced(double x,double y) nogil:
    return sqrt((2.0*x+1.0)*(2.0*y+1.0))

cdef inline double minus_1_pow(double x) nogil:
    cdef int temp = <int>x
    return 1.0 if temp%2 == 0 else -1.0


#### Dipole matrix element between Sigma and Pi states #######################


cpdef double H_int_omega_optimized(state1:SigmaLevel, state2:PiLevelOmega, double pol=0.0):    #pol convention changed. pol defined from ground (state1) to excited (state2).
                                                                    # pol +1 -> mF_state2 - mF_state1 = +1
    cdef double G,N,F1,F,mF,Lambda,Sigma,Omega,Jex,F1p,Fp,mFp

    G,N,F1,F,mF=state1.G,state1.N,state1.F1,state1.F,state1.mF
    Lambda,Sigma,Omega,Jex,F1p,Fp,mFp = state2.Lambda, \
                                        state2.Sigma, \
                                        state2.Omega, \
                                        state2.parity_state.J, \
                                        state2.parity_state.F1, \
                                        state2.parity_state.F, \
                                        state2.parity_state.mF
    
    
    cdef double J,sigma,omega
    cdef int q,iter_idx,i
    cdef double val = 0.0
    cdef double mult_J,mult_sigma,pre_factor

    pre_factor = (minus_1_pow(G+S_+I1_+F-mF+Fp+I2_+F1+1)*
                    sqrt(2*N+1)*
                    wigner_3j(F,1,Fp,-mF,-pol,mFp)*
                    nreduced(F,Fp)*
                    wigner_6j(F1p,Fp,I2_,F,F1,1)*
                    nreduced(F1,F1p)
                    )
    

    for iter_idx in range((int)(2*fabs(N-S_)),(int)(2*(N+S_+1)),2):
        J = 0.5*float(iter_idx)
        mult_J = (nreduced(J,G)*
                    wigner_6j(F1,G,N,S_,J,I1_)*
                    minus_1_pow(F1p+I1_+J+1)*
                    wigner_6j(Jex,F1p,I1_,F1,J,1)*
                    nreduced(J,Jex)
                )

        #for sigma in [-1.0/2,1.0/2]:
        for i in range(-1,2,2):
            sigma = float(i)/2.0
            omega = sigma
            mult_sigma = (mult_J*
                            minus_1_pow(N-S_+omega+J-omega)*wigner_3j(J,S_,N,omega,-sigma,0)*
                            kdel(sigma,Sigma)
                        )
            for q in range(-1,2,2):#[-1,1]: # removed q= 0 value because Lambda (from Pi state) cannot be 0.
                val += mult_sigma*wigner_3j(J,1,Jex,-omega,q,Omega)*kdel(Lambda,-q)
        #iter_idx += 2
    return val*pre_factor

####################################################################################################

In [ ]:
import system
from states import SigmaLevel,PiLevelParity
b=system.System([],[],ignore_mF = False)

g1= SigmaLevel(0.5,0,1/2,1,-1)
g2 = SigmaLevel(0.5,0,1/2,1,1)

e1_p = PiLevelParity(-1,1/2,1/2,0,0)
e1 = PiLevelOmega(1,-1/2,1/2,e1_p)


In [ ]:
%timeit H_int_omega_optimized(g1,e1,0)

2.34 μs ± 136 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
import numpy as np
from numba import njit

@njit(complex128[:, :](int64, complex128[:, :]), cache=True)
def numba_commutator(N, HR):
    comm = np.empty((N, N), dtype=np.complex128)
    for i in range(N):
        # Handle diagonal element
        comm[i, i] = -1j * (HR[i, i] - np.conj(HR[i, i]))
        
        # Compute upper triangular part
        for j in range(i + 1, N):
            val = -1j * (HR[i, j] - np.conj(HR[j, i]))
            comm[i, j] = val
            comm[j, i] = np.conj(val)
    return comm


@njit(complex128[:, :](int64,complex128[:,:]),cache = True,fastmath = True)
def numba_commutator_original(N,HR):
    comm = np.empty((N, N), dtype=np.complex128)
    for i in range(N):
        for j in range(i,N):
            val = -1j * (HR[i, j] - np.conj(HR[j, i]))
            comm[i, j] = val
            if i != j:
                comm[j, i] = np.conj(val)
    return comm

@njit(complex128[:, :](complex128[:, :]), cache=True, fastmath=False)
def numba_commutator_optimized(HR):
    """
    Calculates the communicator using a vectorized approach.
    This version is typically faster and more concise.
    """
    return -1j * (HR - np.conj(HR.T))


@njit(complex128[:, :](int64, complex128[:, :]), cache=True, parallel=True, fastmath=True)
def numba_commutator_parallel(N, HR):
    """
    Optimizes the original loop-based code by running it in parallel.
    """
    comm = np.empty((N, N), dtype=np.complex128)
    # Use prange for a parallel loop
    for i in prange(N):
        for j in range(i, N):
            val = -1j * (HR[i, j] - np.conj(HR[j, i]))
            comm[i, j] = val
            if i != j:
                comm[j, i] = np.conj(val)
    return comm


import time
import matplotlib.pyplot as plt
%matplotlib qt
arr = []
for _ in range(100):
    # Generate a random Hermitian matrix
    N = 1000
    HR = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    HR = HR + HR.conj().T

    # Benchmark original code
    start_time = time.perf_counter()
    comm_original = numba_commutator_original(N, HR)
    end_time = time.perf_counter()
    time_original = end_time - start_time
    #print(f"Original code time: {time_original} seconds")

    # Benchmark updated code
    start_time = time.perf_counter()
    comm_updated = numba_commutator(N, HR)
    end_time = time.perf_counter()
    time_updated = end_time - start_time
    #print(f"Updated code time: {time_updated} seconds")
    #print(f"Update faster by {time_original/time_updated :2f}")

    start_time = time.perf_counter()
    comm_updated = numba_commutator_parallel(N,HR)
    end_time = time.perf_counter()
    time_optimized = end_time - start_time

    arr.append(time_original/time_optimized)
plt.hist(arr,bins=100)

(array([ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,
         1.,  0.,  0.,  0.,  0.,  1.,  0.,  3.,  0.,  1.,  2.,  3.,  2.,
         3.,  2.,  2.,  7.,  8.,  6.,  7.,  4., 10.,  2.,  6.,  5.,  4.,
         1.,  1.,  3.,  1.,  3.,  1.,  1.,  0.,  1.,  0.,  1.,  0.,  1.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.]),
 array([0.16864104, 0.18927215, 0.20990326, 0.23053436, 0.25116547,
        0.27179658, 0.29242769, 0.31305879, 0.3336899 , 0.35432101,
        0.37495211, 0.39558322, 0.41621433, 0.43684544, 0.45747654,
        0.47810765, 0.49873876, 0.51936986, 0.54000097, 0.56063208,
        0.58126319, 0.60189429, 0.6225254 , 0.64315651, 0.66378761,
        0.68441872, 0.70504983, 0.72568093, 0.74631204, 0.76694315,
        0.78757426, 0.8082

In [ ]:
plt.hist(arr,bins=1000)

(array([7.000e+00, 1.000e+01, 8.100e+01, 7.950e+02, 7.832e+03, 4.430e+02,
        7.160e+02, 5.500e+01, 2.900e+01, 1.000e+01, 2.000e+00, 1.000e+00,
        3.000e+00, 2.000e+00, 3.000e+00, 1.000e+00, 2.000e+00, 1.000e+00,
        0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        1.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+0

In [ ]:
import cProfile
import pstats
profiler = cProfile.Profile()
profiler.enable()
numba_commutator_original(N, HR)
profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats('cumtime') # Sort by cumulative time
stats.print_stats()

         129 function calls (128 primitive calls) in 0.007 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.001    0.001 /opt/anaconda3/envs/numpy_boosted/lib/python3.13/site-packages/decorator.py:232(fun)
        1    0.000    0.000    0.000    0.000 /opt/anaconda3/envs/numpy_boosted/lib/python3.13/site-packages/IPython/core/history.py:92(only_when_enabled)
      3/2    0.006    0.002    0.000    0.000 /opt/anaconda3/envs/numpy_boosted/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3631(run_code)
        2    0.000    0.000    0.000    0.000 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 /var/folders/j7/xwzxdxc151n9q90xj3ps384r0000gn/T/ipykernel_8332/2616296526.py:1(<module>)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        2    0.000    0.000    0.000    0.000 /opt/anaconda3/envs/numpy_boos

In [ ]:
%timeit numba_commutator(N,HR)

13.6 μs ± 42 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
%reload_ext cython

In [ ]:
%%cython -a --compile-args=-fopenmp --link-args=fopenmp -lomp
#cython: boundscheck=False, wraparound=False, cdivision=True
import numpy as np
cimport numpy as np
from cython.parallel import prange

DTYPE = np.complex128
ctypedef np.complex128_t DTYPE_t

def matmul(np.ndarray[DTYPE_t, ndim=2] A, np.ndarray[DTYPE_t, ndim=2] B):
    cdef int N = A.shape[0]
    cdef np.ndarray[DTYPE_t, ndim=2] C = np.empty((N, N), dtype=DTYPE)
    
    cdef int i, j, k
    
    for i in prange(N, nogil=True, num_threads=4):
        for j in range(N):
            C[i, j] = 0
            for k in range(N):
                C[i, j] += A[i, k] * B[k, j]
    
    return C

Content of stderr:
clang: error: unsupported option '-fopenmp'

In [ ]:
import numpy as np

N = 100
A = np.random.rand(N, N) + 1j * np.random.rand(N, N)
B = np.random.rand(N, N) + 1j * np.random.rand(N, N)

%timeit matmul(A, B)
%timeit A@B

3.72 ms ± 134 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
73.3 μs ± 3.32 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
